# [Experiment] Vanilla TabNet | Cosine Annealing LR | Forest Cover

### Overview
This notebook evaluates the standard TabNet architecture under a shifted optimization paradigm, utilizing a Cosine Annealing learning rate schedule instead of the traditional StepLR.

### Methodological Context: Continuous Decay
While the structural configuration of the network remains rigorously identical to the original TabNet paper's setup, we are deliberately altering the "thermodynamic" optimization environment. We are departing from the original paper's StepLR schedule to evaluate how the unaltered architecture responds to a smooth, continuous decay, which contrasts sharply with sudden, discrete drops in learning rate.

### The Objective
The goal is to determine how the standard, historically validated architecture responds to a more exploratory and continuously adapting learning rate. This establishes a critical secondary baseline, allowing us to decouple the inherent capabilities of the original TabNet structure from the artifacts of the specific StepLR schedule used by the original authors.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Note: momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
paper_config = {
    'n_d': 64,
    'n_a': 64,
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=0.02),
    'mask_type': 'sparsemax'
}

clf_vanilla = TabNetClassifier(
    **paper_config,
    clip_value=2.,
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params=dict(T_max=MAX_EPOCHS, eta_min=1e-5),
    use_kan=False,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_vanilla.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.15226 | valid_accuracy: 0.03418 |  0:00:08s
epoch 50 | loss: 0.40626 | valid_accuracy: 0.839   |  0:05:52s
epoch 100| loss: 0.15939 | valid_accuracy: 0.93603 |  0:11:38s
epoch 150| loss: 0.22186 | valid_accuracy: 0.92981 |  0:17:24s
epoch 200| loss: 0.1244  | valid_accuracy: 0.94928 |  0:23:10s
epoch 250| loss: 0.08877 | valid_accuracy: 0.9585  |  0:28:55s
epoch 300| loss: 0.07626 | valid_accuracy: 0.96229 |  0:34:42s
epoch 350| loss: 0.07004 | valid_accuracy: 0.96461 |  0:40:28s
epoch 400| loss: 0.08727 | valid_accuracy: 0.9604  |  0:46:14s
epoch 450| loss: 0.06057 | valid_accuracy: 0.96597 |  0:52:01s
epoch 500| loss: 0.06137 | valid_accuracy: 0.96525 |  0:57:47s
epoch 550| loss: 0.05495 | valid_accuracy: 0.96661 |  1:03:34s
epoch 600| loss: 0.0542  | valid_accuracy: 0.96664 |  1:09:19s
epoch 650| loss: 0.04629 | valid_accuracy: 0.96859 |  1:15:07s
epoch 700| loss: 0.04339 | valid_accuracy: 0.96907 |  1:20:53s
epoch 750| loss: 0.04001 | valid_accuracy: 0.96847 |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_vanilla.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 470,580


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_vanilla.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.97088


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "Vanilla TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "CosineAnnealingLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_vanilla.best_epoch,
    "training_history": clf_vanilla.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/03_vanilla_baseline_cosine_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_vanilla.save_model(f'{MODELS_DIR}/03_vanilla_baseline_cosine_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/03_vanilla_baseline_cosine_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/03_vanilla_baseline_cosine_lr_470k.zip
